# Prepare Quartet multiomics inputs for limma correction

The full Quartet matrices are already log-transformed and feature-filtered in the source release. This notebook only aligns each omics layer to metadata, validates the balanced `15 batches x 4 donors x 3 replicates` design, and writes reproducible batch-local inputs.

In [ ]:
library(tidyverse)
library(jsonlite)

WORKDIR <- if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics"
DATA_DIR <- file.path(WORKDIR, "figshare_data")
BEFORE_DIR <- file.path(WORKDIR, "before")
dir.create(BEFORE_DIR, showWarnings = FALSE, recursive = TRUE)

DONOR_LEVELS <- c("D5", "D6", "F7", "M8")

## Dataset registry

Only the full matrices are used. The balanced and confounded subset files are useful for reproducing the paper scenarios, but the selected clustering target is the full four-donor dataset.

In [ ]:
dataset_registry <- tribble(
  ~omics, ~datatype_raw, ~file_name, ~scale,
  "Transcriptomics", "RNA", "Transcriptomics_fulldataset_log2FPKM_r26907c180.csv", "log2(FPKM + 0.01)",
  "Proteomics", "Protein", "Proteomics_fulldataset_log2FOT_r3489c180.csv", "log2(FOT + 0.01)",
  "Metabolomics", "Metabolite", "Metabolomics_fulldataset_log2expr_r71c180.csv", "log2(expr + 1)"
)

dataset_registry

## Loading helpers

Proteomics matrix columns contain periods in a few instrument tokens while metadata uses hyphens, so both sides are normalized before alignment.

In [ ]:
normalize_datatype <- function(x) {
  dplyr::recode(
    as.character(x),
    "RNA" = "Transcriptomics",
    "Protein" = "Proteomics",
    "Metabolite" = "Metabolomics",
    .default = as.character(x)
  )
}

normalize_library_ids <- function(x, omics) {
  x <- as.character(x)
  if (omics == "Proteomics") {
    stringr::str_replace_all(x, stringr::fixed("."), "-")
  } else {
    x
  }
}

safe_batch_name <- function(x) {
  stringr::str_replace_all(as.character(x), "[^A-Za-z0-9_.-]+", "_")
}

load_expression_matrix <- function(path, omics) {
  raw <- readr::read_csv(path, show_col_types = FALSE)
  feature_id_col <- names(raw)[1]
  feature_cols <- feature_id_col
  feature_ids <- as.character(raw[[feature_id_col]])

  if ("metabolite_name" %in% names(raw)) {
    feature_cols <- c(feature_cols, "metabolite_name")
    feature_ids <- paste(feature_ids, raw[["metabolite_name"]], sep = "|")
  }

  expr <- raw %>%
    select(-all_of(feature_cols)) %>%
    as.data.frame(check.names = FALSE)
  rownames(expr) <- make.unique(feature_ids)
  colnames(expr) <- normalize_library_ids(colnames(expr), omics)
  expr[] <- lapply(expr, as.numeric)
  as.matrix(expr)
}

load_full_metadata <- function(path) {
  readr::read_csv(path, show_col_types = FALSE) %>%
    mutate(datatype = normalize_datatype(datatype)) %>%
    group_by(datatype) %>%
    mutate(batch_code = sprintf("B%02d", dense_rank(batch))) %>%
    ungroup() %>%
    transmute(
      file = library,
      condition = sample,
      batch = batch,
      batch_code = batch_code,
      lab = lab,
      platform = platform,
      protocol = replace_na(as.character(protocol), "not_available"),
      datatype = datatype,
      rep = as.integer(rep),
      date = as.character(date)
    )
}

align_expression_to_metadata <- function(expr, metadata, label) {
  missing_in_matrix <- setdiff(metadata$file, colnames(expr))
  extra_in_matrix <- setdiff(colnames(expr), metadata$file)

  if (length(missing_in_matrix) > 0) {
    stop(label, ": metadata samples missing from expression matrix: ",
         paste(head(missing_in_matrix, 10), collapse = ", "), call. = FALSE)
  }
  if (length(extra_in_matrix) > 0) {
    warning(label, ": expression matrix has samples absent from metadata: ",
            paste(head(extra_in_matrix, 10), collapse = ", "))
  }

  metadata <- metadata %>% arrange(match(file, colnames(expr)))
  list(expr = expr[, metadata$file, drop = FALSE], metadata = metadata)
}

validate_full_design <- function(metadata, label) {
  if (nrow(metadata) != 180) stop(label, ": expected 180 samples.", call. = FALSE)
  if (n_distinct(metadata$batch) != 15) stop(label, ": expected 15 batches.", call. = FALSE)
  if (!setequal(unique(metadata$condition), DONOR_LEVELS)) {
    stop(label, ": expected donor levels D5/D6/F7/M8.", call. = FALSE)
  }

  donor_counts <- metadata %>% count(condition)
  if (any(donor_counts$n != 45)) {
    stop(label, ": expected 45 samples per donor.", call. = FALSE)
  }

  batch_donor_counts <- metadata %>% count(batch_code, batch, condition)
  bad <- batch_donor_counts %>% filter(n != 3)
  if (nrow(bad) > 0 || nrow(batch_donor_counts) != 60) {
    stop(label, ": expected exactly 3 replicates per donor in every batch.", call. = FALSE)
  }

  TRUE
}

write_feature_matrix <- function(expr, path) {
  out <- as.data.frame(expr, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out, file = path, sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

write_design_table <- function(metadata, path) {
  design <- metadata %>%
    mutate(
      D6 = as.integer(condition == "D6"),
      F7 = as.integer(condition == "F7"),
      M8 = as.integer(condition == "M8")
    ) %>%
    select(file, D6, F7, M8, batch, batch_code, condition, lab, platform, protocol, datatype, rep, date, pseudo_sample)
  write.table(design, file = path, sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

## Write prepared inputs

Inputs are split by modality and by source batch. No FedRBE configs are written because each Quartet batch has only 12 samples, below the current privacy-filtering requirement for a 15-batch, 3-covariate model.

In [ ]:
full_metadata <- load_full_metadata(file.path(DATA_DIR, "meta_full_dataset_3omics.csv"))

prep_summaries <- purrr::map_dfr(seq_len(nrow(dataset_registry)), function(i) {
  selected <- dataset_registry[i, ]
  omics <- selected$omics
  message("Preparing ", omics)

  expr <- load_expression_matrix(file.path(DATA_DIR, selected$file_name), omics)
  metadata <- full_metadata %>%
    filter(datatype == omics) %>%
    mutate(
      file = normalize_library_ids(file, omics),
      condition = factor(condition, levels = DONOR_LEVELS)
    ) %>%
    arrange(batch_code, condition, rep) %>%
    mutate(
      condition = as.character(condition),
      pseudo_sample = paste(batch_code, condition, rep, sep = "_")
    )

  validate_full_design(metadata, omics)
  aligned <- align_expression_to_metadata(expr, metadata, omics)
  expr <- aligned$expr
  metadata <- aligned$metadata

  variances <- apply(expr, 1, var, na.rm = TRUE)
  stats <- tibble(
    omics = omics,
    features = nrow(expr),
    samples = ncol(expr),
    batches = n_distinct(metadata$batch),
    donors = n_distinct(metadata$condition),
    missing_cells = sum(is.na(expr)),
    rows_with_any_na = sum(rowSums(is.na(expr)) > 0),
    zero_or_na_variance_rows = sum(!is.finite(variances) | variances == 0)
  )

  if (stats$missing_cells > 0) stop(omics, ": source matrix contains missing values.", call. = FALSE)
  if (stats$zero_or_na_variance_rows > 0) stop(omics, ": source matrix contains zero-variance rows.", call. = FALSE)
  if (anyDuplicated(metadata$pseudo_sample) > 0) stop(omics, ": duplicate pseudo-sample keys.", call. = FALSE)

  modality_dir <- file.path(BEFORE_DIR, omics)
  dir.create(modality_dir, showWarnings = FALSE, recursive = TRUE)

  write_feature_matrix(expr, file.path(modality_dir, "central_intensities_log_UNION.tsv"))
  write.table(metadata, file = file.path(modality_dir, "metadata.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

  batch_lookup <- metadata %>% distinct(batch, batch_code) %>% mutate(batch_dir = safe_batch_name(batch))
  if (anyDuplicated(batch_lookup$batch_dir) > 0) stop(omics, ": duplicate safe batch folder names.", call. = FALSE)

  for (batch_value in batch_lookup$batch) {
    batch_meta <- metadata %>% filter(batch == batch_value) %>% arrange(condition, rep)
    batch_dir <- file.path(modality_dir, safe_batch_name(batch_value))
    dir.create(batch_dir, showWarnings = FALSE, recursive = TRUE)

    write_feature_matrix(expr[, batch_meta$file, drop = FALSE], file.path(batch_dir, "intensities_log_UNION.tsv"))
    write_design_table(batch_meta, file.path(batch_dir, "design.tsv"))
  }

  stats
})

prep_summaries

## Save preparation summary

The summary records the matrix shape and basic validation checks used by the central correction notebook.

In [ ]:
write.table(prep_summaries, file = file.path(BEFORE_DIR, "preparation_summary.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

datainfo <- list(
  source = "Quartet full multiomics Figshare 10.6084/m9.figshare.22188349.v1",
  sample_set = "full four-donor dataset",
  correction_batch = "batch",
  preserved_biology = DONOR_LEVELS,
  modalities = prep_summaries
)
writeLines(jsonlite::toJSON(datainfo, pretty = TRUE, auto_unbox = TRUE), file.path(BEFORE_DIR, "datainfo.json"))

cat("Prepared inputs written to", normalizePath(BEFORE_DIR), "\n")

## Session information

In [ ]:
sessionInfo()